# Apple ML-SHARP — persistent model worker

目标：把 SHARP 的 **checkpoint 下载**、**模型构造/加载**、**full-size warmup** 从每次请求中彻底拿掉。

```text
setup / clone
    │
    ▼
download_model.py       # 只下载 checkpoint；存在就直接复用
    │
    ▼
/kaggle/working/models/sharp/sharp_2572gikvuh.pt
    │
    ▼
model_runtime.py        # create_predictor + load_state_dict + cuda + warmup
    │                   # 只执行一次，模型常驻 GPU
    ▼
model_server.py         # localhost:7861，1 worker
    │
    ▼
app.py                  # 纯 Gradio + HTTP，不 import torch/sharp
```

**效果：**
- 重启 `app.py`：不会重新加载 SHARP。
- 重复运行 Worker Cell：如果 `7861/health` 已就绪，直接复用。
- 只有 `model_server.py` 被终止、Kernel 重启、Kaggle Session 重启时才重新加载模型。
- SHARP 官方内部推理尺寸保持 `1536×1536`。
- 输入文件原始字节直接传给 Worker，尽量保留 EXIF 焦距；无焦距时沿用官方默认逻辑。

> 注意：Apple 发布的 SHARP 模型权重许可证限定为 Research Purposes（非商业科研/学术开发）。本 Notebook 仅按该研究用途设计。


## 1. 安装运行环境

不替换 Kaggle 自带 PyTorch；只安装 SHARP 推理所需依赖和 UI/HTTP 依赖。

In [2]:
!python -m pip install -q -U uv

from pathlib import Path
import subprocess

REPO = Path("/kaggle/working/ml-sharp")

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/apple/ml-sharp.git", str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

!uv pip install --system \
    "gradio>=5" \
    "gradio-tunneling" \
    "fastapi>=0.115" \
    "uvicorn>=0.30" \
    "python-multipart>=0.0.9" \
    "requests>=2.32" \
    "pillow==11.3.0" \
    "pillow-heif>=1.1" \
    "plyfile>=1.1" \
    "scipy>=1.14" \
    "timm>=1.0.20" \
    "matplotlib>=3.9"

!uv pip install --system --no-deps -e /kaggle/working/ml-sharp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 71.9 MB/s eta 0:00:00:00:0100:01


Cloning into '/kaggle/working/ml-sharp'...


Using Python 3.12.13 environment at: /usr
Resolved 91 packages in 911ms                                        
Prepared 3 packages in 534ms                                             
Installed 3 packages in 7ms9.0                              
 + gradio-tunneling==0.9.0
 + pillow-heif==1.5.0
 + plyfile==1.1.5
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 3ms                                            
Prepared 1 package in 1.29s                                              
Installed 1 package in 1msle:///kaggle/working/ml-sharp)    
 + sharp==0.1 (from file:///kaggle/working/ml-sharp)


## 2. 检查 GPU / PyTorch / SHARP

In [3]:
import sys
from pathlib import Path

# 将 ml-sharp 的 src 目录加入 Python 搜索路径
sharp_src = "/kaggle/working/ml-sharp/src"
if sharp_src not in sys.path:
    sys.path.insert(0, sharp_src)

import torch
import PIL
import sharp

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("Pillow:", PIL.__version__)
print("SHARP:", sharp.__file__)
print("GPU count:", torch.cuda.device_count())
print("GPU 0:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

assert torch.cuda.is_available(), "没有检测到 GPU，请先在 Kaggle Notebook Settings 打开 GPU。"

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA: 12.8
Pillow: 11.3.0
SHARP: /kaggle/working/ml-sharp/src/sharp/__init__.py
GPU count: 2
GPU 0: Tesla T4


## 3. 预下载 SHARP checkpoint

官方 CLI 在没有传 `-c` 时会通过 `torch.hub.load_state_dict_from_url(...)` 自动下载。

这里改成显式下载到固定目录：

```text
/kaggle/working/models/sharp/sharp_2572gikvuh.pt
```

以后 `model_runtime.py` 只读取本地文件，不在模型启动时下载。

In [4]:
%%writefile download_model.py
from pathlib import Path
import os
import requests

URL = "https://ml-site.cdn-apple.com/models/sharp/sharp_2572gikvuh.pt"
ROOT = Path("/kaggle/working/models/sharp")
CHECKPOINT = ROOT / "sharp_2572gikvuh.pt"
PART = CHECKPOINT.with_suffix(".pt.part")

ROOT.mkdir(parents=True, exist_ok=True)

if CHECKPOINT.exists() and CHECKPOINT.stat().st_size > 0:
    print(
        f"SHARP checkpoint already exists -> {CHECKPOINT} "
        f"({CHECKPOINT.stat().st_size / 2**20:.1f} MiB)"
    )
else:
    if PART.exists():
        PART.unlink()

    print("Downloading:", URL)
    with requests.get(URL, stream=True, timeout=(20, 600)) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0

        with PART.open("wb") as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if total:
                    print(
                        f"\r{done / 2**20:.1f} / {total / 2**20:.1f} MiB "
                        f"({done / total * 100:.1f}%)",
                        end="",
                        flush=True,
                    )

    os.replace(PART, CHECKPOINT)
    print()

print(
    f"SHARP checkpoint READY -> {CHECKPOINT} "
    f"({CHECKPOINT.stat().st_size / 2**20:.1f} MiB)"
)


Writing download_model.py


In [5]:
!python download_model.py

Downloading: https://ml-site.cdn-apple.com/models/sharp/sharp_2572gikvuh.pt
2679.6 / 2679.6 MiB (100.0%)
SHARP checkpoint READY -> /kaggle/working/models/sharp/sharp_2572gikvuh.pt (2679.6 MiB)


## 4. 常驻 GPU Runtime

这个模块只会在 `model_server.py` 进程第一次 import 时执行：

1. 从本地 `.pt` 读取权重
2. `create_predictor(PredictorParams())`
3. `load_state_dict`
4. `.eval().to("cuda:0")`
5. 用官方 `1536×1536` 内部尺寸做一次真实 forward warmup
6. 后续请求永远复用同一个 `PREDICTOR`

为了避免 T4 上并发请求叠加显存，整个 `image → 3DGS → PLY` 流程用一把锁串行化。

In [6]:
%%writefile model_runtime.py
import gc
import threading
import time
from pathlib import Path

import torch
import torch.nn.functional as F

from sharp.models import PredictorParams, create_predictor
from sharp.utils import io
from sharp.utils.gaussians import save_ply, unproject_gaussians

ROOT = Path("/kaggle/working")
CHECKPOINT = ROOT / "models/sharp/sharp_2572gikvuh.pt"

if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f"缺少 SHARP checkpoint: {CHECKPOINT}\n"
        "请先运行 download_model.py"
    )

if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 CUDA GPU")

DEVICE = torch.device("cuda:0")
INTERNAL_SHAPE = (1536, 1536)
MODEL_LOCK = threading.Lock()

torch.backends.cudnn.benchmark = True


def sync():
    torch.cuda.synchronize(DEVICE)


def gpu_status():
    free, total = torch.cuda.mem_get_info(DEVICE)
    allocated = torch.cuda.memory_allocated(DEVICE)
    reserved = torch.cuda.memory_reserved(DEVICE)
    return (
        f"{DEVICE} | {torch.cuda.get_device_name(DEVICE)}\n"
        f"allocated={allocated / 2**30:.2f} GiB | "
        f"reserved={reserved / 2**30:.2f} GiB | "
        f"free={free / 2**30:.2f}/{total / 2**30:.2f} GiB"
    )


# ============================================================
# Load once
# ============================================================

print("\n" + "=" * 72)
print("Loading Apple SHARP from LOCAL checkpoint -> cuda:0")
print("=" * 72)

t0 = time.perf_counter()

state_dict = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=True,
)

PREDICTOR = create_predictor(PredictorParams())
PREDICTOR.load_state_dict(state_dict)
del state_dict

PREDICTOR = PREDICTOR.eval().to(DEVICE)
sync()

MODEL_LOAD_SECONDS = time.perf_counter() - t0
print(f"SHARP model loaded: {MODEL_LOAD_SECONDS:.2f}s")
print(gpu_status())


# ============================================================
# Warmup once
# ============================================================

print("\n" + "=" * 72)
print("Full-size warmup: 1 × 3 × 1536 × 1536")
print("=" * 72)

t0 = time.perf_counter()

dummy = torch.zeros(
    (1, 3, INTERNAL_SHAPE[1], INTERNAL_SHAPE[0]),
    dtype=torch.float32,
    device=DEVICE,
)
dummy_disparity_factor = torch.ones((1,), dtype=torch.float32, device=DEVICE)

with torch.inference_mode():
    warm_output = PREDICTOR(dummy, dummy_disparity_factor)

sync()
MODEL_WARMUP_SECONDS = time.perf_counter() - t0

del warm_output, dummy, dummy_disparity_factor
gc.collect()

print(f"SHARP warmup: {MODEL_WARMUP_SECONDS:.2f}s")
print("\n" + "=" * 72)
print("SHARP READY — predictor stays resident on cuda:0")
print("=" * 72)
print(gpu_status())
print("=" * 72)


# ============================================================
# Official-equivalent prediction path
# ============================================================

@torch.inference_mode()
def _predict_image(image, f_px):
    t_total = time.perf_counter()

    # ---------- preprocess ----------
    t = time.perf_counter()

    image_pt = (
        torch.from_numpy(image.copy())
        .float()
        .to(DEVICE)
        .permute(2, 0, 1)
        / 255.0
    )

    _, height, width = image_pt.shape

    disparity_factor = torch.tensor(
        [f_px / width],
        dtype=torch.float32,
        device=DEVICE,
    )

    image_resized_pt = F.interpolate(
        image_pt[None],
        size=(INTERNAL_SHAPE[1], INTERNAL_SHAPE[0]),
        mode="bilinear",
        align_corners=True,
    )

    preprocess_s = time.perf_counter() - t

    # ---------- network forward ----------
    sync()
    t = time.perf_counter()

    gaussians_ndc = PREDICTOR(image_resized_pt, disparity_factor)

    sync()
    inference_s = time.perf_counter() - t

    # ---------- official metric-space postprocess ----------
    t = time.perf_counter()

    intrinsics = (
        torch.tensor(
            [
                [f_px, 0, width / 2, 0],
                [0, f_px, height / 2, 0],
                [0, 0, 1, 0],
                [0, 0, 0, 1],
            ],
            dtype=torch.float32,
            device=DEVICE,
        )
    )

    intrinsics_resized = intrinsics.clone()
    intrinsics_resized[0] *= INTERNAL_SHAPE[0] / width
    intrinsics_resized[1] *= INTERNAL_SHAPE[1] / height

    gaussians = unproject_gaussians(
        gaussians_ndc,
        torch.eye(4, device=DEVICE),
        intrinsics_resized,
        INTERNAL_SHAPE,
    )

    postprocess_s = time.perf_counter() - t
    total_s = time.perf_counter() - t_total

    del image_pt, image_resized_pt, disparity_factor, gaussians_ndc
    del intrinsics, intrinsics_resized

    return gaussians, {
        "preprocess_s": preprocess_s,
        "inference_s": inference_s,
        "postprocess_s": postprocess_s,
        "predict_total_s": total_s,
    }


def run_sharp(input_path, output_path):
    input_path = Path(input_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # 官方 load_rgb：
    # - 保留图像 EXIF 逻辑
    # - 尝试读取 35mm 等效焦距
    # - 找不到时默认 30mm
    image, _, f_px = io.load_rgb(input_path)

    with MODEL_LOCK:
        t_total = time.perf_counter()

        gaussians, timing = _predict_image(image, f_px)

        t = time.perf_counter()
        save_ply(
            gaussians,
            f_px,
            (image.shape[0], image.shape[1]),
            output_path,
        )
        save_s = time.perf_counter() - t

        timing["save_ply_s"] = save_s
        timing["request_total_s"] = time.perf_counter() - t_total

        del gaussians
        gc.collect()

    return {
        "output_path": str(output_path),
        "width": int(image.shape[1]),
        "height": int(image.shape[0]),
        "focal_px": float(f_px),
        "timing": timing,
        "gpu_status": gpu_status(),
    }


Writing model_runtime.py


## 5. localhost 模型 Worker

`model_server.py` 只有 **1 个 uvicorn worker**。

多 worker 会复制多份 SHARP 模型，所以这里明确禁止。

In [7]:
%%writefile model_server.py
from pathlib import Path
from uuid import uuid4

from fastapi import FastAPI, File, HTTPException, UploadFile

# import 时：
#   1. 加载一次 checkpoint
#   2. 构造一次 predictor
#   3. warmup 一次
# server 存活期间不会重复。
from model_runtime import (
    MODEL_LOAD_SECONDS,
    MODEL_WARMUP_SECONDS,
    gpu_status,
    run_sharp,
)

ROOT = Path("/kaggle/working")
INPUT_DIR = ROOT / "sharp-inputs"
OUTPUT_DIR = ROOT / "sharp-outputs"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(title="Persistent Apple SHARP Worker")


@app.get("/health")
def health():
    return {
        "ready": True,
        "model": "apple/ml-sharp",
        "model_load_s": MODEL_LOAD_SECONDS,
        "model_warmup_s": MODEL_WARMUP_SECONDS,
        "gpu_status": gpu_status(),
    }


@app.post("/infer")
def infer(file: UploadFile = File(...)):
    filename = file.filename or "input.png"
    suffix = Path(filename).suffix.lower() or ".png"
    stem = Path(filename).stem or "input"

    token = uuid4().hex[:12]
    input_path = INPUT_DIR / f"{stem}-{token}{suffix}"
    output_path = OUTPUT_DIR / f"{stem}-{token}.ply"

    try:
        input_path.write_bytes(file.file.read())
        if input_path.stat().st_size == 0:
            raise ValueError("上传文件为空")

        result = run_sharp(input_path, output_path)
        result["original_name"] = filename
        return result

    except Exception as e:
        try:
            output_path.unlink(missing_ok=True)
        except Exception:
            pass
        raise HTTPException(status_code=500, detail=f"SHARP inference failed: {e}")

    finally:
        try:
            input_path.unlink(missing_ok=True)
        except Exception:
            pass


if __name__ == "__main__":
    import uvicorn

    uvicorn.run(
        app,
        host="127.0.0.1",
        port=7861,
        workers=1,
        log_level="info",
    )


Writing model_server.py


## 6. 纯 Gradio UI

这个进程不 import `torch`，也不 import `sharp`。

它只做：

```text
上传原始图片
   │
   ▼
HTTP POST -> localhost:7861/infer
   │
   ▼
常驻 SHARP Worker
   │
   ├─ 返回 .ply 路径
   └─ 返回耗时 / GPU 状态
```


In [8]:
%%writefile app.py
from functools import lru_cache
from pathlib import Path
from urllib.parse import quote
import math
import gradio as gr
import numpy as np
import requests
import uvicorn
from fastapi import FastAPI, HTTPException, Query
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from plyfile import PlyData

WORKER = "http://127.0.0.1:7861"
TIMEOUT = 600
OUTPUT_DIR = Path("/kaggle/working/sharp-outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def worker_status():
    try:
        r = requests.get(WORKER + "/health", timeout=3)
        r.raise_for_status()
        data = r.json()
        return f"Worker READY\nmodel load: {data['model_load_s']:.2f}s\nwarmup: {data['model_warmup_s']:.2f}s\n{data['gpu_status']}"
    except Exception as e:
        return f"Worker NOT READY\n{e}"

@lru_cache(maxsize=32)
def _read_scene_info_cached(file_path: str, mtime_ns: int):
    path = Path(file_path)
    ply = PlyData.read(str(path))
    vertex = ply["vertex"].data
    x = np.asarray(vertex["x"], dtype=np.float64)
    y = np.asarray(vertex["y"], dtype=np.float64)
    z = np.asarray(vertex["z"], dtype=np.float64)
    finite = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
    x = x[finite]; y = y[finite]; z = z[finite]
    if len(x) == 0: raise RuntimeError("PLY 中没有有效 Gaussian 坐标")
    xyz_min = np.array([x.min(), y.min(), z.min()], dtype=np.float64)
    xyz_max = np.array([x.max(), y.max(), z.max()], dtype=np.float64)
    center = (xyz_min + xyz_max) * 0.5
    extent = xyz_max - xyz_min
    radius = float(np.linalg.norm(extent) * 0.5)
    if not math.isfinite(radius) or radius <= 1e-6: radius = 1.0
    return {"count": int(len(x)), "min": xyz_min.tolist(), "max": xyz_max.tolist(), "center": center.tolist(), "extent": extent.tolist(), "radius": radius, "file_mb": path.stat().st_size / (1024 ** 2)}

def read_scene_info(path: Path):
    stat = path.stat()
    return _read_scene_info_cached(str(path), stat.st_mtime_ns)

def waiting_viewer():
    return '<div style="height:680px;border:1px solid var(--border-color-primary);border-radius:10px;display:flex;align-items:center;justify-content:center;font-size:16px;opacity:.72;">生成完成后，这里会显示 SHARP 原始 3D Gaussian Splat</div>'

def viewer_iframe(filename: str):
    src = "/viewer?file=" + quote(filename)
    return f'<iframe src="{src}" style="width:100%;height:700px;border:1px solid rgba(127,127,127,.35);border-radius:10px;background:#101010;" allow="fullscreen"></iframe>'

def run_sharp(image_path, progress=gr.Progress()):
    if not image_path: raise gr.Error("请先上传图片")
    path = Path(image_path)
    progress(0.03, desc="提交到常驻 SHARP Worker...")
    try:
        with path.open("rb") as f:
            r = requests.post(WORKER + "/infer", files={"file": (path.name, f, "application/octet-stream")}, timeout=TIMEOUT)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        raise gr.Error(f"SHARP Worker 调用失败:\n{e}")
    progress(0.88, desc="SHARP 推理完成，检查 Gaussian PLY...")
    output_path = Path(data["output_path"])
    if not output_path.exists(): raise gr.Error(f"Worker 返回的 PLY 不存在:\n{output_path}")
    try:
        scene = read_scene_info(output_path)
    except Exception as e:
        raise gr.Error(f"PLY 已生成，但无法解析场景范围:\n{e}")
    progress(0.96, desc="计算场景范围并初始化 3D Viewer...")
    t = data["timing"]
    cx, cy, cz = scene["center"]
    minx, miny, minz = scene["min"]
    maxx, maxy, maxz = scene["max"]
    info = (f"输入: {data['width']}×{data['height']}\n焦距: {data['focal_px']:.2f}px\n\nGaussians: {scene['count']:,}\nPLY: {scene['file_mb']:.1f} MiB\n\nscene center:\n  x={cx:.4f}\n  y={cy:.4f}\n  z={cz:.4f}\n\nscene z range:\n  {minz:.4f} → {maxz:.4f}\n\nscene radius: {scene['radius']:.4f}\n\npreprocess:        {t['preprocess_s']:.3f}s\nnetwork inference: {t['inference_s']:.3f}s\npostprocess:       {t['postprocess_s']:.3f}s\nsave PLY:          {t['save_ply_s']:.3f}s\nrequest total:     {t['request_total_s']:.3f}s\n\n{data['gpu_status']}")
    progress(1.0, desc="完成")
    return (str(output_path), viewer_iframe(output_path.name), info, worker_status())

with gr.Blocks(title="Apple ML-SHARP — Persistent Worker + Spark Viewer") as demo:
    gr.Markdown("""# Apple ML-SHARP — Persistent GPU Worker\n\n模型常驻：\n`model_server.py :7861`\n\nUI / Viewer：\n`app.py :7860`\n\n```text\nImage\n  ↓\nSHARP\n  ↓\nMetric 3D Gaussians\n  ↓\nPLY\n  ↓\nSpark / WebGL\n```""")
    with gr.Row():
        with gr.Column(scale=1):
            image = gr.File(label="输入图片（保留原始文件 / EXIF）", file_types=["image"], type="filepath")
            run_btn = gr.Button("生成并可视化 3DGS", variant="primary")
            output = gr.File(label="下载 SHARP 原始 3DGS (.ply)")
        with gr.Column(scale=2):
            viewer = gr.HTML(value=waiting_viewer(), label="Spark 3D Gaussian Viewer")
    with gr.Accordion("Benchmark / Scene / GPU", open=True):
        with gr.Row():
            info = gr.Textbox(label="Benchmark / Scene", lines=20)
            status = gr.Textbox(label="Worker / GPU", value=worker_status(), lines=8)
    gr.Markdown("""\n### Viewer 操作\n\n* 左键拖动：旋转\n* 右键拖动：平移\n* 滚轮：缩放\n* `Reset Camera`：恢复自动计算的最佳视角\n* `Front`：回到 SHARP 输入相机方向\n* `Axes`：显示 / 隐藏坐标轴\n\n注意：生成结束后，浏览器还需要下载一次 `.ply`。\n如果 PLY 是 60 MiB，Viewer 的加载不算在 `request total` 中。""")
    run_btn.click(run_sharp, inputs=image, outputs=[output, viewer, info, status])

web = FastAPI(title="SHARP + Spark Gaussian Viewer")
web.mount("/outputs", StaticFiles(directory=str(OUTPUT_DIR)), name="sharp-outputs")

@web.get("/viewer", response_class=HTMLResponse)
def splat_viewer(file: str = Query(...)):
    filename = Path(file).name
    path = OUTPUT_DIR / filename
    if not path.exists() or path.suffix.lower() != ".ply":
        raise HTTPException(status_code=404, detail="PLY not found")
    try:
        scene = read_scene_info(path)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"PLY parse error: {e}")
    cx, cy, cz = scene["center"]
    radius = scene["radius"]
    minx, miny, minz = scene["min"]
    maxx, maxy, maxz = scene["max"]
    count = scene["count"]
    file_mb = scene["file_mb"]
    distance = max(radius * 2.6, 0.5)
    camera_x = cx; camera_y = cy; camera_z = cz - distance
    near = max(radius / 1000.0, 0.001)
    far = max(radius * 50.0, distance * 10.0, 100.0)
    ply_url = "/outputs/" + quote(filename)
    page = f"""<!doctype html>
<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>SHARP Gaussian Splat</title>
<style>html,body{{width:100%;height:100%;margin:0;padding:0;overflow:hidden;background:#101010;color:#eee;font-family:system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;}}
#canvas{{position:absolute;inset:0;width:100%;height:100%;touch-action:none;}}
#toolbar{{position:absolute;z-index:20;top:12px;left:12px;display:flex;gap:7px;flex-wrap:wrap;}}
button{{border:1px solid rgba(255,255,255,.22);border-radius:7px;padding:7px 11px;background:rgba(18,18,18,.78);color:#eee;cursor:pointer;backdrop-filter:blur(8px);}}
button:hover{{background:rgba(55,55,55,.92);}}
#status{{position:absolute;z-index:20;left:12px;bottom:12px;max-width:min(520px,calc(100% - 24px));padding:10px 12px;border-radius:8px;background:rgba(0,0,0,.67);backdrop-filter:blur(8px);font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace;font-size:12px;line-height:1.55;white-space:pre-wrap;pointer-events:none;}}
#error{{position:absolute;display:none;z-index:100;inset:0;align-items:center;justify-content:center;padding:30px;background:#101010;color:#ffb4b4;white-space:pre-wrap;font-family:ui-monospace,monospace;overflow:auto;}}
</style>
<script type="importmap">{{"imports":{{"three":"https://cdn.jsdelivr.net/npm/three@0.180.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.180.0/examples/jsm/","@sparkjsdev/spark":"https://sparkjs.dev/releases/spark/2.1.0/spark.module.js"}}}}</script>
</head><body>
<canvas id="canvas"></canvas>
<div id="toolbar"><button id="reset">Reset Camera</button><button id="front">Front</button><button id="axes">Axes</button></div>
<div id="status">Initializing Spark...</div>
<div id="error"></div>
<script type="module">
import * as THREE from "three";
import {{OrbitControls}} from "three/addons/controls/OrbitControls.js";
import {{SparkRenderer,SplatMesh}} from "@sparkjsdev/spark";
const canvas=document.getElementById("canvas");
const status=document.getElementById("status");
const errorBox=document.getElementById("error");
const meta={{count:{count},fileMB:{file_mb},center:new THREE.Vector3({cx},{cy},{cz}),radius:{radius},min:new THREE.Vector3({minx},{miny},{minz}),max:new THREE.Vector3({maxx},{maxy},{maxz})}};
const scene=new THREE.Scene();
scene.background=new THREE.Color(0x101010);
const camera=new THREE.PerspectiveCamera(55,window.innerWidth/window.innerHeight,{near},{far});
camera.up.set(0,-1,0);
const defaultCameraPosition=new THREE.Vector3({camera_x},{camera_y},{camera_z});
camera.position.copy(defaultCameraPosition);
camera.lookAt(meta.center);
const renderer=new THREE.WebGLRenderer({{canvas:canvas,antialias:false,powerPreference:"high-performance"}});
renderer.setPixelRatio(Math.min(window.devicePixelRatio,2));
renderer.setSize(window.innerWidth,window.innerHeight);
const controls=new OrbitControls(camera,renderer.domElement);
controls.target.copy(meta.center);
controls.enableDamping=true;
controls.dampingFactor=0.08;
controls.screenSpacePanning=true;
controls.minDistance=Math.max(meta.radius*0.03,0.01);
controls.maxDistance=Math.max(meta.radius*20,10);
controls.update();
const spark=new SparkRenderer({{renderer:renderer}});
scene.add(spark);
const axes=new THREE.AxesHelper(Math.max(meta.radius*0.5,0.25));
axes.position.copy(meta.center);
axes.visible=false;
scene.add(axes);
function statusText(extra=""){{const p=camera.position;status.textContent=`SHARP 3D Gaussian Splat\n\nPLY\n  Gaussian:${{meta.count.toLocaleString()}}\n  File:     ${{meta.fileMB.toFixed(1)}} MiB\n\nScene\n  center:\n    ${{meta.center.x.toFixed(4)}}\n    ${{meta.center.y.toFixed(4)}}\n    ${{meta.center.z.toFixed(4)}}\n\n  radius:\n    ${{meta.radius.toFixed(4)}}\n\nCamera\n  ${{p.x.toFixed(3)}}\n  ${{p.y.toFixed(3)}}\n  ${{p.z.toFixed(3)}}\n\n${{extra}}`;}}
function fail(err){{console.error(err);errorBox.style.display="flex";errorBox.textContent="Spark Viewer 加载失败\\n\\n"+(err?.stack||err?.message||String(err));}}
function resetCamera(){{camera.position.copy(defaultCameraPosition);camera.up.set(0,-1,0);controls.target.copy(meta.center);camera.lookAt(meta.center);controls.update();statusText("Camera reset");}}
function frontView(){{const distance=Math.max(meta.radius*2.6,0.5);camera.position.set(meta.center.x,meta.center.y,meta.center.z-distance);controls.target.copy(meta.center);camera.up.set(0,-1,0);camera.lookAt(meta.center);controls.update();statusText("Front view");}}
document.getElementById("reset").onclick=resetCamera;
document.getElementById("front").onclick=frontView;
document.getElementById("axes").onclick=()=>{{axes.visible=!axes.visible;}};
window.addEventListener("resize",()=>{{camera.aspect=window.innerWidth/window.innerHeight;camera.updateProjectionMatrix();renderer.setSize(window.innerWidth,window.innerHeight);}});
try{{statusText("Downloading / decoding PLY...");
const splat=new SplatMesh({{url:"{ply_url}",onProgress:(event)=>{{if(event&&event.total&&event.total>0){{const pct=event.loaded/event.total*100;statusText("Loading PLY: "+pct.toFixed(1)+"%");}}else{{statusText("Loading PLY...");}}}}}});
scene.add(splat);
await splat.initialized;
statusText("✓ PLY loaded\\n✓ Spark initialized\\n✓ Camera auto-fitted");
}}catch(err){{fail(err);}}
renderer.setAnimationLoop(()=>{{controls.update();renderer.render(scene,camera);}});
</script></body></html>"""
    return HTMLResponse(page)

demo.queue(default_concurrency_limit=2)
web = gr.mount_gradio_app(web, demo, path="/", allowed_paths=[str(OUTPUT_DIR)])

if __name__ == "__main__":
    uvicorn.run(web, host="127.0.0.1", port=7860, workers=1, log_level="info")

Overwriting app.py


## 7. 启动常驻 SHARP Worker

这个 Cell 可以重复运行：

- `7861` 已经健康：直接复用，**不重新 load / warmup**
- `7861` 不存在：才启动 `model_server.py`

第一次启动会看到模型加载和 full-size warmup 的耗时。

In [9]:
import subprocess
import sys
import time
from pathlib import Path

import requests

WORKER_URL = "http://127.0.0.1:7861/health"
WORKER_LOG_PATH = Path("/kaggle/working/sharp_model_server.log")


def worker_ready():
    try:
        r = requests.get(WORKER_URL, timeout=1)
        return r.ok and r.json().get("ready") is True
    except Exception:
        return False


if worker_ready():
    print("✅ 复用现有 SHARP model_server.py；不会重新加载模型")
else:
    WORKER_LOG = open(WORKER_LOG_PATH, "a", buffering=1)

    MODEL_SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "model_server.py"],
        stdout=WORKER_LOG,
        stderr=subprocess.STDOUT,
        cwd="/kaggle/working",
    )

    while not worker_ready():
        if MODEL_SERVER_PROCESS.poll() is not None:
            WORKER_LOG.flush()
            print(WORKER_LOG_PATH.read_text(errors="replace")[-12000:])
            raise RuntimeError(
                f"model_server.py 已退出，code={MODEL_SERVER_PROCESS.returncode}"
            )
        time.sleep(1)

    print("✅ SHARP model_server.py READY")

health = requests.get(WORKER_URL, timeout=3).json()
print(f"model load: {health['model_load_s']:.2f}s")
print(f"warmup: {health['model_warmup_s']:.2f}s")
print(health["gpu_status"])


✅ SHARP model_server.py READY
model load: 13.77s
warmup: 50.66s
cuda:0 | Tesla T4
allocated=2.63 GiB | reserved=4.96 GiB | free=9.47/14.56 GiB


## 8. 启动 / 重启 Gradio UI

只重启 `app.py`。

**不会终止 `7861`，所以不会重新加载 SHARP。**

In [10]:
import subprocess
import sys
import time
from pathlib import Path

import requests

APP_LOG_PATH = Path("/kaggle/working/sharp_app.log")

if "APP_PROCESS" in globals():
    try:
        if APP_PROCESS.poll() is None:
            APP_PROCESS.terminate()
            APP_PROCESS.wait(timeout=5)
    except Exception:
        try:
            APP_PROCESS.kill()
        except Exception:
            pass

APP_LOG = open(APP_LOG_PATH, "a", buffering=1)

APP_PROCESS = subprocess.Popen(
    [sys.executable, "app.py"],
    stdout=APP_LOG,
    stderr=subprocess.STDOUT,
    cwd="/kaggle/working",
)

while True:
    if APP_PROCESS.poll() is not None:
        APP_LOG.flush()
        print(APP_LOG_PATH.read_text(errors="replace")[-12000:])
        raise RuntimeError(f"app.py 已退出，code={APP_PROCESS.returncode}")

    try:
        r = requests.get("http://127.0.0.1:7860", timeout=1)
        if r.ok:
            break
    except Exception:
        pass

    time.sleep(0.5)

print("✅ Gradio ready: http://127.0.0.1:7860")
print("✅ SHARP GPU 模型仍常驻 7861，不会因 UI 重启而重载")


✅ Gradio ready: http://127.0.0.1:7860
✅ SHARP GPU 模型仍常驻 7861，不会因 UI 重启而重载


## 9. Kaggle Tunnel

In [ ]:
!gradio-tun 7860

公网访问地址：https://1c7adf9f94b44c22df.gradio.live
这个共享链接将在 72 小时后过期，此程序将在 72 小时后关闭。


## 运行时结构

```text
Kaggle Session
│
├─ model_server.py :7861
│   │
│   ├─ CHECKPOINT（本地）
│   ├─ PREDICTOR（cuda:0 常驻）
│   ├─ full-size warmup（只做一次）
│   └─ MODEL_LOCK
│       │
│       └─ image
│           → resize 1536²
│           → SHARP forward
│           → NDC → metric 3D
│           → save .ply
│
└─ app.py :7860
    │
    ├─ 上传原始图片
    ├─ HTTP -> :7861
    └─ 返回 .ply + benchmark
```

### 这里刻意没有做的事

- **不在每次请求里执行 `sharp predict` CLI**  
  因为 CLI 每次都会重新构造并加载 predictor。

- **不让 Gradio import SHARP / torch**  
  UI 可以随时重启，不影响 GPU resident model。

- **不启多个 uvicorn worker**  
  否则每个 worker 都会复制一套 GPU 模型。

- **默认不做 `--render`**  
  生成 `.ply` 和 gsplat 视频渲染是两条不同的性能路径。先把 SHARP 主模型 resident 化，避免 gsplat 初始化/编译干扰主推理 benchmark。
